# Project 2 – Using custom Python Class: SparkDataCheck

## Part 1: Testing our Class on Air Quality Data

### Introduction

In this notebook we'll use our custom `SparkDataCheck` Python class. To start, we'll use it on the **UCI Air Quality dataset** from project 1, which records hourly averaged
sensor measurements of several pollutants (CO, NOx, NO₂, Benzene, O₃, etc.)in an Italian city over roughly one year.


1. **Setup** – import the class and start a Spark session.
2. **Load from CSV via the class** – use `SparkDataCheck.from_csv()` to read the
   data directly into a `SparkDataCheck` object.
3. **Validation method demos** – 4–5 examples of each validation method
   (`check_numeric_bounds`, `check_categorical_levels`, `check_missing`), including
   cases that trigger warning messages (non-numeric column, missing bounds, etc.).
4. **Summarization method demos** – 4–5 examples of each summarization method
   (`summarize_numeric`, `summarize_counts`).
5. **Load from pandas** – read the same CSV with pandas, then create a second
   `SparkDataCheck` instance via `SparkDataCheck.from_pandas()`.
6. **Single method call on the pandas-sourced object**.


### Setup

First, we will reload the python class, import other needed modules, and start a spark session.


In [7]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Project2") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

In [17]:
import importlib
import my_class
importlib.reload(my_class)

<module 'my_class' from '/home/jupyter-ramerser@ncsu.edu/my_class.py'>

### Load Data 

Let's read the air quality CSV directly from the csv filepath into a `SparkDataCheck`
object using the `from_csv()` class method.  

The dataset contains **9,357 rows** and **16 columns**. Missing values are recorded as
`-200` rather than `NULL`, which we will look at later on.


In [21]:
CSV_PATH = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

air_pandas = pd.read_csv(CSV_PATH)
air_pandas.to_csv("air.csv", index=False)

# Load from local path
air_check = my_class.SparkDataCheck.from_csv(spark, "air.csv")

# Quick look at the schema and first few rows
air_check.df.printSchema()
air_check.df.show(5, truncate=False)


root
 |-- Unnamed: 0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)

+----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|Date     |Time               |CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|T   |RH  |AH    |
+----------+---------+

###  Validation Methods

Now that the data is loaded in correctly, lets check some of our validation methods!

First, lets look at  

#### `check_numeric_range()`

This method appends a Boolean column indicating whether each
value falls within the supplied range (inclusive). NULL values will appear as NULL.

**Example 1 – Both bounds, CO sensor:** Acceptable CO(GT) readings should sit
between 0 and 11 mg/m³, so we'll see if the first 10 observations fall within this range.


In [23]:
# Example 1: both lower and upper bounds on a numeric column
air_check.check_numeric_range("CO(GT)", lower=0, upper=11)
air_check.df.select("CO(GT)", "CO(GT)_in_bounds").show(10)


+------+----------------+
|CO(GT)|CO(GT)_in_bounds|
+------+----------------+
|   2.6|            true|
|   2.0|            true|
|   2.2|            true|
|   2.2|            true|
|   1.6|            true|
|   1.2|            true|
|   1.2|            true|
|   1.0|            true|
|   0.9|            true|
|   0.6|            true|
+------+----------------+
only showing top 10 rows


**Example 2 – Lower bound only:** Check whether temperature (`T`) is at least −5°C
(i.e., flag any suspiciously cold readings).


In [24]:
# Example 2: lower bound only
air_check.check_numeric_range("T", lower=-5)
air_check.df.select("T", "T_in_bounds").show(10)


+----+-----------+
|   T|T_in_bounds|
+----+-----------+
|13.6|       true|
|13.3|       true|
|11.9|       true|
|11.0|       true|
|11.2|       true|
|11.2|       true|
|11.3|       true|
|10.7|       true|
|10.7|       true|
|10.3|       true|
+----+-----------+
only showing top 10 rows


**Example 3 – Upper bound only:** Relative humidity (`RH`) should never exceed 100%.


In [25]:
# Example 3: upper bound only
air_check.check_numeric_range("RH", upper=100)
air_check.df.select("RH", "RH_in_bounds").show(10)


+----+------------+
|  RH|RH_in_bounds|
+----+------------+
|48.9|        true|
|47.7|        true|
|54.0|        true|
|60.0|        true|
|59.6|        true|
|59.2|        true|
|56.8|        true|
|60.0|        true|
|59.7|        true|
|60.2|        true|
+----+------------+
only showing top 10 rows


**Example 4 – No bounds supplied (triggers warning message):**
Calling the method without providing *either* bound should print an informative
message and leave the DataFrame unchanged.


In [26]:
# Example 4: neither bound provided – should print a warning
air_check.check_numeric_range("NOx(GT)")


check_numeric_range: please supply at least one of 'lower' or 'upper'.


**Example 5 – Non-numeric column supplied (triggers warning message):**
Passing the string-typed `Date` column should print a message and return the
object unmodified.


In [27]:
# Example 5: non-numeric column – should print a type warning
air_check.check_numeric_range("Date", lower=0, upper=100)
print("Columns after invalid call:", air_check.df.columns)


check_numeric_bounds: column 'Date' has type 'string' which is not numeric. DataFrame is unchanged.
Columns after invalid call: ['Unnamed: 0', 'Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'CO(GT)_in_bounds', 'T_in_bounds', 'RH_in_bounds']


Now, let's look at 5 examples of our next method:

#### `check_levels()`

This method appends a Boolean column indicating whether each value in a *string*
column belongs to a defined set of levels. We can use the `Date` and `Time` columns to
demonstrate this method.

**Example 1 – Valid months in the Date column:**
The dataset runs from March 2004 to February 2005. We check whether the month
portion of the date falls within a known good set.


In [33]:
# Example 1: check Date values against a set of known dates in the dataset
# Add a Month string column so we have a clean categorical to check
air_check.df = air_check.df.withColumn("Month", F.split(F.col("Date"), "/")[0])

valid_months = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]
air_check.check_levels("Month", levels=valid_months)
air_check.df.select("Date", "Month", "Month_in_levels").show(10)


+---------+-----+---------------+
|     Date|Month|Month_in_levels|
+---------+-----+---------------+
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/10/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
|3/11/2004|    3|           true|
+---------+-----+---------------+
only showing top 10 rows


**Example 2 – Subset of valid months (some will be False):**
Only flag dates from the summer months (June–August) as valid.


In [34]:
# Example 2: restrict to summer months – many rows will be False
air_check.check_levels("Month", levels=["6","7","8"])
air_check.df.select("Date", "Month", "Month_in_levels").show(10)


+---------+-----+---------------+
|     Date|Month|Month_in_levels|
+---------+-----+---------------+
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/10/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
|3/11/2004|    3|          false|
+---------+-----+---------------+
only showing top 10 rows


**Example 3 – Check Time values against expected hours:**
The data is hourly; valid hours run 00–23.


In [36]:
# Example 3: check Time column against expected hour strings
air_check.df = air_check.df.withColumn("Time", F.date_format(F.col("Time"), "HH:mm:ss"))

valid_hours = [f"{h:02d}:00:00" for h in range(24)]
air_check.check_levels("Time", levels=valid_hours)
air_check.df.select("Time", "Time_in_levels").show(10)


+--------+--------------+
|    Time|Time_in_levels|
+--------+--------------+
|18:00:00|          true|
|19:00:00|          true|
|20:00:00|          true|
|21:00:00|          true|
|22:00:00|          true|
|23:00:00|          true|
|00:00:00|          true|
|01:00:00|          true|
|02:00:00|          true|
|03:00:00|          true|
+--------+--------------+
only showing top 10 rows


**Example 4 – Non-string column supplied (triggers warning message):**
Passing the integer column `PT08.S1(CO)` should print a message and return the
object unmodified.


In [37]:
# Example 4: non-string column – should print a type warning
air_check.check_levels("PT08.S1(CO)", levels=["good","bad"])


check_categorical_levels: column 'PT08.S1(CO)' has type 'int' which is not a string. DataFrame is unchanged.


**Example 5 – Method chaining across two categorical checks:**
We can chain the call on `Month` and `Time` in a single expression because each
method returns `self`.


In [38]:
# Example 5: chained categorical checks
air_check \
    .check_levels("Month", levels=["6","7","8"]) \
    .check_levels("Time", levels=["12.00.00","13.00.00","14.00.00"])

air_check.df.select("Date","Time","Month_in_levels","Time_in_levels").show(10)


+---------+--------+---------------+--------------+
|     Date|    Time|Month_in_levels|Time_in_levels|
+---------+--------+---------------+--------------+
|3/10/2004|18:00:00|          false|         false|
|3/10/2004|19:00:00|          false|         false|
|3/10/2004|20:00:00|          false|         false|
|3/10/2004|21:00:00|          false|         false|
|3/10/2004|22:00:00|          false|         false|
|3/10/2004|23:00:00|          false|         false|
|3/11/2004|00:00:00|          false|         false|
|3/11/2004|01:00:00|          false|         false|
|3/11/2004|02:00:00|          false|         false|
|3/11/2004|03:00:00|          false|         false|
+---------+--------+---------------+--------------+
only showing top 10 rows


Now, lets look at 5 examples of

#### `check_missing()` 

This method appends a Boolean column that is `True` wherever the source column is
`NULL`. It works on *any* column type.


##### Brief Data Cleaning

Since null values are recorded as `-200` in this data, let's replace them with null to accurately use our `check_missing()` method


In [49]:
air_check.df = air_check.df.na.replace(-200.0, None)
air_check.df = air_check.df.na.replace(-200, None)

# To confirm it worked, this should show a null
air_check.df.select("CO(GT)", "NOx(GT)", "T").show(10)

+------+-------+----+
|CO(GT)|NOx(GT)|   T|
+------+-------+----+
|   2.6|    166|13.6|
|   2.0|    103|13.3|
|   2.2|    131|11.9|
|   2.2|    172|11.0|
|   1.6|    131|11.2|
|   1.2|     89|11.2|
|   1.2|     62|11.3|
|   1.0|     62|10.7|
|   0.9|     45|10.7|
|   0.6|   NULL|10.3|
+------+-------+----+
only showing top 10 rows


**Example 1 – Missing CO sensor readings:**

In [50]:
# Example 1: check for NULLs in CO(GT)
air_check.check_missing("CO(GT)")
air_check.df.select("CO(GT)", "CO(GT)_is_missing").show(10)


+------+-----------------+
|CO(GT)|CO(GT)_is_missing|
+------+-----------------+
|   2.6|            false|
|   2.0|            false|
|   2.2|            false|
|   2.2|            false|
|   1.6|            false|
|   1.2|            false|
|   1.2|            false|
|   1.0|            false|
|   0.9|            false|
|   0.6|            false|
+------+-----------------+
only showing top 10 rows


**Example 2 – Missing temperature readings:**


In [51]:
# Example 2: missing values in T (temperature)
air_check.check_missing("T")
air_check.df.select("T", "T_is_missing").show(10)


+----+------------+
|   T|T_is_missing|
+----+------------+
|13.6|       false|
|13.3|       false|
|11.9|       false|
|11.0|       false|
|11.2|       false|
|11.2|       false|
|11.3|       false|
|10.7|       false|
|10.7|       false|
|10.3|       false|
+----+------------+
only showing top 10 rows


**Example 3 – Missing values in a string column (`Date`):**
The method accepts any column type, so we can use it on `Date` as well.


In [52]:
# Example 3: missing Date values
air_check.check_missing("Date")
air_check.df.select("Date", "Date_is_missing").show(10)


+---------+---------------+
|     Date|Date_is_missing|
+---------+---------------+
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/10/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
|3/11/2004|          false|
+---------+---------------+
only showing top 10 rows


**Example 4 – Count the proportion of missing values for a column:**
Once the Boolean column exists, standard Spark aggregation can tell us the
missing rate. This is the kind of user-level analysis the class deliberately leaves
to the user.


In [53]:
# Example 4: compute missing rate for AH (absolute humidity)
air_check.check_missing("AH")
from pyspark.sql import functions as F

total = air_check.df.count()
missing = air_check.df.filter(F.col("AH_is_missing")).count()
print(f"AH: {missing}/{total} rows are NULL ({100*missing/total:.1f}%)")


AH: 366/9357 rows are NULL (3.9%)


**Example 5 – Chaining `check_missing` with `check_numeric_bounds`:**
We can seamlessly mix validation methods in a single chain.


In [54]:
# Example 5: chain missing-check with a bounds-check on NOx
air_check \
    .check_missing("NOx(GT)") \
    .check_numeric_range("NOx(GT)", lower=0, upper=1000)

air_check.df.select("NOx(GT)", "NOx(GT)_is_missing", "NOx(GT)_in_bounds").show(10)


+-------+------------------+-----------------+
|NOx(GT)|NOx(GT)_is_missing|NOx(GT)_in_bounds|
+-------+------------------+-----------------+
|    166|             false|             true|
|    103|             false|             true|
|    131|             false|             true|
|    172|             false|             true|
|    131|             false|             true|
|     89|             false|             true|
|     62|             false|             true|
|     62|             false|             true|
|     45|             false|             true|
|   NULL|              true|             NULL|
+-------+------------------+-----------------+
only showing top 10 rows


###  Summarization Methods

####   `summarize_numeric()` – 5 examples

Returns a **pandas** DataFrame (not Spark) with the min and max of numeric
column(s). An optional `group_by` argument adds a grouping dimension.

**Example 1 – Single column, no grouping:**


In [55]:
# Example 1: min/max of CO(GT) – no grouping
result = air_check.summarize_numeric("CO(GT)")
print(type(result))
result


<class 'pandas.core.frame.DataFrame'>


,CO(GT)_min,CO(GT)_max
0,0.1,11.9


**Example 2 – Single column grouped by Month:**


In [ ]:
# Example 2: min/max of temperature grouped by month
result = air_check.summarize_numeric("T", group_by="Month")
result


**Example 3 – All numeric columns, no grouping:**
When no column is supplied the method finds and summarizes every numeric column
in a single wide pandas DataFrame.


In [ ]:
# Example 3: all numeric columns at once
result = air_check.summarize_numeric()
result


**Example 4 – All numeric columns grouped by Month (`reduce` + `pd.merge` path):**


In [ ]:
# Example 4: all numeric columns grouped by month
result = air_check.summarize_numeric(group_by="Month")
result


**Example 5 – Non-numeric column supplied (triggers warning, returns None):**


In [ ]:
# Example 5: non-numeric column – should print a warning and return None
result = air_check.summarize_numeric("Date")
print("Return value:", result)


####  `summarize_counts()` – 5 examples

Returns a **pandas** DataFrame of value counts for one or two string columns.

**Example 1 – Counts by Month:**


In [ ]:
# Example 1: row counts per month
result = air_check.summarize_counts("Month")
result


**Example 2 – Counts by Time (hour of day):**


In [ ]:
# Example 2: row counts per hour
result = air_check.summarize_counts("Time")
result


**Example 3 – Cross-tabulation: Month × Time:**
With two string columns, every combination of levels is counted — essentially a
cross-tabulation.


In [ ]:
# Example 3: counts for Month × Time combinations
result = air_check.summarize_counts("Month", column2="Time")
print(result.shape)
result.head(15)


**Example 4 – Non-string primary column (triggers warning, returns None):**


In [ ]:
# Example 4: numeric column as primary – should warn and return None
result = air_check.summarize_counts("CO(GT)")
print("Return value:", result)


**Example 5 – Non-string secondary column (triggers warning, returns None):**


In [ ]:
# Example 5: valid string primary, but numeric secondary – should warn
result = air_check.summarize_counts("Month", column2="T")
print("Return value:", result)


###  Load from Pandas – `from_pandas()`

Now we read the **same dataset** using plain pandas, then hand the pandas DataFrame
to `SparkDataCheck.from_pandas()`. This path calls `spark.createDataFrame()` to
convert the pandas DataFrame into a Spark DataFrame before wrapping it in our class.

This is useful when data has already been loaded or pre-processed in pandas and you
want to run the Spark-based quality checks without re-reading from disk.


In [ ]:
# Read with pandas first
air_df = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
print("Pandas shape:", air_df.shape)
air_df.head()


In [ ]:
# Create a SparkDataCheck instance from the pandas DataFrame
air_check_pd = SparkDataCheck.from_pandas(spark, air_df)

# Confirm the Spark DataFrame was created correctly
air_check_pd.df.printSchema()


###  Example Method Call on the Pandas-Sourced Object

We run `summarize_numeric()` on `air_check_pd` to check the min/max of **all
numeric columns** — a quick sanity check that the pandas-to-Spark conversion
preserved the data correctly. We expect results that match what we saw from the
CSV-sourced object above.


In [ ]:
# Single example: min/max of all numeric columns from the pandas-sourced object
summary_pd = air_check_pd.summarize_numeric()
summary_pd


## Part 2: NFL Data Analysis